# 第一部分：数据获取

本notebook用于下载10只股票和2个指数的行情数据，以及宏观和财务指标数据。

In [ ]:
import pandas as pd
import numpy as np
import akshare as ak
import baostock as bs
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 创建目录结构
dirs = ['data/stock', 'data/index', 'data/macro', 'data/finance', 'data/clean', 'data/combined', 'output']
for d in dirs:
    os.makedirs(d, exist_ok=True)
print('目录结构创建完成')

## 1.1 下载股票行情数据

In [ ]:
# 股票列表
stocks = [
    ('600036', '招商银行', '银行'),
    ('601398', '工商银行', '银行'),
    ('601127', '赛力斯', '汽车'),
    ('000625', '长安汽车', '汽车'),
    ('600048', '保利发展', '房地产'),
    ('001979', '招商蛇口', '房地产'),
    ('600519', '贵州茅台', '白酒'),
    ('000568', '泸州老窖', '白酒'),
    ('601012', '隆基绿能', '能源'),
    ('300274', '阳光电源', '能源')
]

# 使用baostock下载后复权数据
bs.login()
log_entries = []

for code, name, industry in stocks:
    try:
        rs = bs.query_history_k_data_plus(code, 
            'date,open,high,low,close,volume,amount',
            start_date='2020-01-01', end_date='2026-05-21',
            frequency='d', adjustflag='2')
        data_list = []
        while rs.error_code == '0' and rs.next():
            data_list.append(rs.get_row_data())
        if data_list:
            df = pd.DataFrame(data_list, columns=['日期', '开盘', '收盘', '最高', '最低', '成交量', '成交额'])
            df['股票代码'] = code
            df.to_csv(f'data/stock/{code}.csv', index=False)
            log_entries.append(f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] SUCCESS  stock_{code}  shape={df.shape}')
            print(f'SUCCESS: {name}({code}) - {len(df)} rows')
        else:
            log_entries.append(f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] FAILED  stock_{code}  Error: No data returned')
            print(f'FAILED: {name}({code}) - No data')
    except Exception as e:
        log_entries.append(f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] FAILED  stock_{code}  Error: {str(e)}')
        print(f'FAILED: {name}({code}) - {e}')

bs.logout()

## 1.2 下载市场指数

In [ ]:
# 沪深300
df_hs300 = ak.stock_zh_index_daily(symbol='sh000300')
df_hs300 = df_hs300[df_hs300['date'] >= '2020-01-01']
df_hs300.to_csv('data/index/index_000300.csv', index=False)
log_entries.append(f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] SUCCESS  index_000300  shape={df_hs300.shape}')
print(f'沪深300: {len(df_hs300)} rows')

# 中证500
df_zz500 = ak.stock_zh_index_daily(symbol='sh000905')
df_zz500 = df_zz500[df_zz500['date'] >= '2020-01-01']
df_zz500.to_csv('data/index/index_000905.csv', index=False)
log_entries.append(f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] SUCCESS  index_000905  shape={df_zz500.shape}')
print(f'中证500: {len(df_zz500)} rows')

## 1.3 下载宏观指标

In [ ]:
# CPI
df_cpi = ak.macro_china_cpi_monthly()
df_cpi.to_csv('data/macro/macro_cpi.csv', index=False)
log_entries.append(f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] SUCCESS  macro_cpi  shape={df_cpi.shape}')
print(f'CPI: {len(df_cpi)} rows')

# M2
df_m2 = ak.macro_china_m2_monthly()
df_m2.to_csv('data/macro/macro_m2.csv', index=False)
log_entries.append(f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] SUCCESS  macro_m2  shape={df_m2.shape}')
print(f'M2: {len(df_m2)} rows')

## 1.4 保存下载日志

In [ ]:
with open('download_log.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_entries))
print('下载日志已保存到 download_log.txt')